# CalmFruits — неделя 2: каталог и поисковые индексы

Тетрадь готовит каталог MVP, фиксирует общий каталог оценки и создаёт лексический и семантический индексы. Она не читает тестовую разметку и не оценивает качество.

In [1]:
from __future__ import annotations

import importlib.metadata
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists(): ROOT = ROOT.parent
assert (ROOT / 'src' / 'calmfruits').exists()
sys.path.insert(0, str(ROOT / 'src'))

from calmfruits.catalog import MVP_ROOT_CATEGORIES, load_week2_sources, prepare_catalog, save_catalogs
from calmfruits.data import load_local_env
from calmfruits.search import LexicalSearch, SemanticSearch, SEMANTIC_BATCH_SIZE, SEMANTIC_MODEL, SEMANTIC_REVISION, save_index_manifest

SEED = 42
np.random.seed(SEED)
CATALOG_DIR = ROOT / 'artifacts' / 'catalog'
INDEX_DIR = ROOT / 'artifacts' / 'indexes'
assert load_local_env(ROOT / '.env'), 'S3 credentials are required in local .env'
environment = pd.DataFrame({'value': {
    'python': sys.version.split()[0], 'pandas': importlib.metadata.version('pandas'),
    'scikit_learn': importlib.metadata.version('scikit-learn'),
    'sentence_transformers': importlib.metadata.version('sentence-transformers'),
    'torch': importlib.metadata.version('torch'), 'seed': SEED, 'device': 'CPU',
    'mvp_categories': ', '.join(MVP_ROOT_CATEGORIES), 'semantic_model': SEMANTIC_MODEL,
    'semantic_batch_size': SEMANTIC_BATCH_SIZE, 'semantic_revision': SEMANTIC_REVISION, 'source_files': 'raw catalog, train relevance, dedup imt_id',
}})
display(environment)


,value
python,3.12.6
pandas,2.2.3
scikit_learn,1.6.1
sentence_transformers,5.5.1
torch,2.12.0
seed,42
device,CPU
mvp_categories,"Одежда, Обувь"
semantic_model,sentence-transformers/paraphrase-multilingual-...
semantic_batch_size,32


## 1. Реальные источники и подготовка каталога

Тексты берём из сырого каталога. Служебный `wb_products_dedup` применяется только как список идентификаторов общего каталога оценки.

In [2]:
products_raw, queries_train, evaluation_ids = load_week2_sources()
assert {'imt_id', 'nm_id', 'imt_name', 'description', 'subj_name', 'subj_root_name'}.issubset(products_raw.columns)
assert evaluation_ids.notna().all() and evaluation_ids.is_unique
full_catalog, evaluation_catalog, catalog_stages = prepare_catalog(products_raw, evaluation_ids)
assert evaluation_catalog.imt_id.is_unique
assert set(evaluation_catalog.imt_id).issubset(set(full_catalog.imt_id))
save_catalogs(full_catalog, evaluation_catalog, catalog_stages, CATALOG_DIR)
display(catalog_stages)
display(evaluation_catalog[['imt_id', 'imt_name', 'subj_root_name', 'subj_name', 'product_text']].head())


,stage,rows,unique_imt_id
0,raw_rows,200000,174427
1,selected_root_categories,56503,48489
2,required_clean_text,22271,17967
3,deduplicated_imt_id,17967,17967
4,evaluation_catalog_intersection,3280,3280


,imt_id,imt_name,subj_root_name,subj_name,product_text
0,100211,Жилет,Одежда,Жилеты,Жилет . Жилеты . Привлекательный удлиненный жи...
1,1000021,Куртка,Одежда,Куртки,Куртка . Куртки . Прекрасная куртка с застежко...
2,1000024,Джемперы,Одежда,Джемперы,Джемперы . Джемперы . Замечательный легкий дже...
3,1000027,Джемпер,Одежда,Джемперы,Джемпер . Джемперы . Великолепный джемпер прит...
4,1000087,Брюки,Одежда,Брюки,Брюки . Брюки . Стильные брюки для истинных мо...


**Вывод по `catalog_stages`.** Из 200 000 исходных строк в корнях «Одежда» и «Обувь» осталось 56 503. После строгой проверки очищенных `imt_name` и `description` осталось 22 271 строка; детерминированная дедупликация дала 17 967 карточек `imt_id`. Пересечение со служебным списком оценки содержит 3 280 карточек — это единый каталог обоих поисков.

## 2. Проверка очистки и дедупликации

Показываем десять выбранных карточек до и после очистки. Полные тексты сохраняются в отдельный HTML-артефакт, чтобы notebook не обрезал их при отображении.

In [3]:
before_after = evaluation_catalog.sample(n=min(10, len(evaluation_catalog)), random_state=SEED).copy()
raw_before = products_raw.loc[before_after.source_row, ['imt_id', 'nm_id', 'imt_name', 'description']].reset_index(drop=True)
after_cleaning = before_after[['imt_id', 'nm_id', 'imt_name', 'description', 'product_text']].reset_index(drop=True)
before_after_table = raw_before.merge(after_cleaning, on=['imt_id', 'nm_id'], suffixes=('_raw', '_clean'), validate='one_to_one')
before_after_path = CATALOG_DIR / 'before_after_cleaning_examples.html'
before_after_table.to_html(before_after_path, index=False, escape=True)
assert before_after_table.imt_id.nunique() == len(before_after_table)
assert not evaluation_catalog.product_text.str.contains(r'(?:^| \. )(?:None|nan)(?: \. |$)', case=False, regex=True).any()
print(f'Full before/after examples: {before_after_path}')
display(before_after_table[['imt_id', 'nm_id', 'imt_name_raw', 'imt_name_clean', 'product_text']])
dedup_check = full_catalog.groupby('imt_id').size().max()
assert dedup_check == 1


Full before/after examples: /Users/dkunicyn/Documents/to-future/calm-fruit/artifacts/catalog/before_after_cleaning_examples.html


,imt_id,nm_id,imt_name_raw,imt_name_clean,product_text
0,10168678,13579117,Сарафан,Сарафан,Сарафан . Сарафаны . Для выбора наиболее подхо...
1,10133366,13531882,Толстовка,Толстовка,Толстовка . Толстовки . Толстовка с капюшоном ...
2,10085631,13467419,Кигуруми комбинезон детская / Пижама для сна,Кигуруми комбинезон детская / Пижама для сна,Кигуруми комбинезон детская / Пижама для сна ....
3,1007223,1189022,Брюки,Брюки,Брюки . Брюки . Стильные брюки классического к...
4,10108055,13498879,Ветровка большого размера демисезонная,Ветровка большого размера демисезонная,Ветровка большого размера демисезонная . Куртк...
5,10004418,13355910,Платье,Платье,Платье . Платья . Платье из трикотажного италь...
6,10044870,13412157,Мужская обувь/е кроссовки/мужские изики,Мужская обувь/е кроссовки/мужские изики,Мужская обувь/е кроссовки/мужские изики . Крос...
7,10165872,13575613,Бриджи женские трикотажные большие размеры/спо...,Бриджи женские трикотажные большие размеры/спо...,Бриджи женские трикотажные большие размеры/спо...
8,10013005,13367145,Юбка,Юбка,"Юбка . Юбки . Состав Вверх - 100% Полиэстер, П..."
9,10096870,13483567,Зуавы,Зуавы,Зуавы . Бриджи . Легкая и практичная Российска...


**Вывод по `before_after_table` и `dedup_check`.** Полные десять примеров сохранены в `before_after_cleaning_examples.html`; они подтверждают удаление HTML и нормализацию пробелов без удаления технических обозначений. Assertion `dedup_check == 1` подтверждает ровно одну строку на `imt_id`, а проверка `product_text` исключает строковые артефакты `None`/`nan`.

## 3. Лексический и семантический индексы

Оба индекса строятся по одному и тому же `evaluation_catalog`; семантическая модель кодирует товарные тексты батчами и нормализует векторы.

In [4]:
lexical = LexicalSearch.fit(evaluation_catalog)
lexical.save(INDEX_DIR)
semantic = SemanticSearch.fit(evaluation_catalog)
semantic.save(INDEX_DIR)
save_index_manifest(evaluation_catalog, INDEX_DIR)
assert lexical.matrix.shape[0] == len(evaluation_catalog)
assert semantic.embeddings.shape == (len(evaluation_catalog), 384)
assert np.allclose(np.linalg.norm(semantic.embeddings, axis=1), 1, atol=1e-4)
product_truncation = semantic.truncation_statistics(evaluation_catalog.product_text.tolist())
query_truncation = semantic.truncation_statistics(queries_train[['query_id', 'query_text']].drop_duplicates().query_text.tolist())
index_inventory = pd.DataFrame([
    {'index': 'tfidf', 'shape': str(lexical.matrix.shape), 'vocabulary_size': len(lexical.vectorizer.vocabulary_)},
    {'index': 'semantic', 'shape': str(semantic.embeddings.shape), 'vocabulary_size': np.nan},
])
display(index_inventory)
display(pd.concat({'products': product_truncation, 'queries_train': query_truncation}))
display(pd.DataFrame(json.loads((INDEX_DIR / 'index_manifest.json').read_text())).T)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (176 > 128). Running this sequence through the model will result in indexing errors


,index,shape,vocabulary_size
0,tfidf,"(3280, 50000)",50000.0
1,semantic,"(3280, 384)",NaN


,,texts,max_seq_length,median_tokens,p95_tokens,max_tokens,truncated_share
products,0,3280,128,102.0,325.05,1136,0.366159
queries_train,0,700,128,8.0,12.00,18,0.000000


,catalog_size,imt_id_order,product_text_columns,root_categories,ngram_range,max_features,lowercase,norm
catalog,3280,"[100211, 1000021, 1000024, 1000027, 1000087, 1...","[imt_name, subj_name, description]","[Одежда, Обувь]",NaN,NaN,NaN,NaN
tfidf,NaN,NaN,NaN,NaN,"[1, 2]",50000,True,l2
semantic_model,sentence-transformers/paraphrase-multilingual-...,sentence-transformers/paraphrase-multilingual-...,sentence-transformers/paraphrase-multilingual-...,sentence-transformers/paraphrase-multilingual-...,sentence-transformers/paraphrase-multilingual-...,sentence-transformers/paraphrase-multilingual-...,sentence-transformers/paraphrase-multilingual-...,sentence-transformers/paraphrase-multilingual-...
semantic_revision,e8f8c211226b894fcb81acc59f3b34ba3efd5f42,e8f8c211226b894fcb81acc59f3b34ba3efd5f42,e8f8c211226b894fcb81acc59f3b34ba3efd5f42,e8f8c211226b894fcb81acc59f3b34ba3efd5f42,e8f8c211226b894fcb81acc59f3b34ba3efd5f42,e8f8c211226b894fcb81acc59f3b34ba3efd5f42,e8f8c211226b894fcb81acc59f3b34ba3efd5f42,e8f8c211226b894fcb81acc59f3b34ba3efd5f42
semantic_batch_size,32,32,32,32,32,32,32,32


**Вывод по `index_inventory`, `product_truncation` и `query_truncation`.** TF-IDF построен как матрица 3 280×50 000, семантический индекс — 3 280×384. Модель имеет лимит 128 токенов: усечению подвергаются 36,62% товарных текстов, но ни один train-запрос; это ограничение фиксируется для дальнейших экспериментов.

## 4. Контракт поиска и сохранённые артефакты

Проверяем пустой запрос, неизвестные лексические токены, детерминированный порядок, большой и некорректный `top_k`, а также повторную загрузку артефактов.

In [5]:
from joblib import load
from scipy import sparse
from sentence_transformers import SentenceTransformer
from calmfruits.search import RESULT_COLUMNS

assert lexical.search_lexical('   ', 5).empty
assert semantic.search_semantic('', 5).empty
unknown = lexical.search_lexical('неизвестноесловоxyz', len(evaluation_catalog) + 10)
assert len(unknown) == len(evaluation_catalog) and unknown.imt_id.is_monotonic_increasing
for engine in (lexical.search_lexical, semantic.search_semantic):
    for invalid in (0, -1, 1.2, True):
        try: engine('кроссовки', invalid)
        except ValueError: pass
        else: raise AssertionError(f'invalid top_k accepted: {invalid!r}')
    result = engine('кожаные кроссовки', 5)
    assert result.columns.tolist() == RESULT_COLUMNS
    assert result['rank'].tolist() == [1, 2, 3, 4, 5]
    assert result.imt_id.is_unique
reloaded_lexical = LexicalSearch(evaluation_catalog, load(INDEX_DIR / 'tfidf_vectorizer.joblib'), sparse.load_npz(INDEX_DIR / 'tfidf_matrix.npz'))
reloaded_semantic = SemanticSearch(evaluation_catalog, SentenceTransformer(SEMANTIC_MODEL, revision=SEMANTIC_REVISION, device='cpu'), np.load(INDEX_DIR / 'semantic_embeddings.npy'))
assert reloaded_lexical.search_lexical('кожаные кроссовки', 10).equals(lexical.search_lexical('кожаные кроссовки', 10))
assert reloaded_semantic.search_semantic('кожаные кроссовки', 10).equals(semantic.search_semantic('кожаные кроссовки', 10))
display(lexical.search_lexical('кожаные кроссовки', 5))
display(semantic.search_semantic('кожаные кроссовки', 5))


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

,rank,imt_id,score,imt_name,subj_name,description
0,1,10131000,0.361573,Кроссовки / кеды кожаные,Кроссовки,Кроссовки МАЛОМЕРЯТ на один размер! Удобная и ...
1,2,10046686,0.183365,Кроссовки женские,Кроссовки,"Кроссовки женские, из текстильного материала. ..."
2,3,10047162,0.181649,Кроссовки,Кроссовки,Кроссовки GOGC TREND выполнены из текстиля. Мо...
3,4,10038574,0.180923,Кроссовки,Кроссовки,Стильные кроссовки выполнены из экокожи. Кросс...
4,5,10085215,0.180292,Кроссовки,Кроссовки,Стильные кроссовки выполнены из искусственной ...


,rank,imt_id,score,imt_name,subj_name,description
0,1,1013366,0.870566,Ботинки,Ботинки,Отличные кожаные ботинки с подкладкой из искус...
1,2,10109662,0.869290,Кроссовки кожаные повседневные 201725-1-1101,Полуботинки,Восхитительные и невероятно удобные полуботинк...
2,3,10109654,0.844776,Кеды кожаные туфли повседневные 201707-4-1101,Полуботинки,Восхитительные и невероятно удобные полуботинк...
3,4,10109648,0.839552,Полуботинки кожаные туфли классические 201702-...,Полуботинки,Восхитительные и невероятно удобные полуботинк...
4,5,10109653,0.819577,Кеды кожаные классические 201707-3-1301,Полуботинки,Восхитительные и невероятно удобные полуботинк...


**Вывод по контрактным assertions и выдачам.** Пустые запросы возвращают пустую таблицу, неизвестный TF-IDF-запрос возвращает весь каталог в порядке `imt_id` при равных score, а некорректный `top_k` отклоняется. Повторная загрузка сохранённых TF-IDF и embedding-артефактов воспроизводит top-10 обеих систем.